## Streaming and Inference

**Inference:** the model generating a response to an input. Happens after training is complete.
**Streaming:** sending each token to the user as it is generated, instead of waiting for the full response.

## How it actually works

**Inference:**
Training is done. Model is deployed. A user sends input — the model processes it and generates output.
This is the only side of the model you interact with as an API consumer.
Every time you call the Claude API — that is inference.

**Two types of inference:**
- **Online inference:** real-time, one request at a time, user waiting for immediate response
- **Batch inference:** process many inputs together at a scheduled time, no real-time pressure

**Streaming:**
LLMs generate text one token at a time.
In non-streaming mode: model generates all tokens, then sends the complete response.
In streaming mode: each token is sent to the client immediately as it is generated.
The user sees text appearing word by word — like watching someone type.

**Why streaming matters:**
A 500-word response takes 10-15 seconds to generate.
Without streaming: user stares at a blank screen for 15 seconds.
With streaming: user starts reading after the first token — under 1 second.
Same total time. Completely different experience.

## Where it breaks or gets dangerous

**Use streaming for:**
- Chat interfaces — user expects to see responses appearing
- Long responses — nobody waits 30 seconds for a blank screen
- RAG assistants — retrieval + generation takes time, streaming masks the wait

**Do NOT use streaming for:**
- Structured JSON output — you need the complete response before parsing
- Pipeline steps — if the next step needs the full output, streaming adds complexity
- Batch processing — no human watching, streaming adds overhead with no benefit

**Common mistake:**
Using streaming everywhere by default.
A batch invoice processor streaming responses to nobody is wasted complexity.
Match the inference type to the use case.

**Batch API advantage:**
Anthropic's Batch API processes requests asynchronously at 50% lower cost.
For nightly fraud scans, weekly reports, bulk content generation — always use batch.
Never use real-time inference for work that doesn't need to be real-time.

In [ ]:
# Streaming vs Non-Streaming — conceptual illustration
import anthropic
import os
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

print("=== NON-STREAMING ===")
print("User waits... waits... waits... then sees complete response at once")
print()

# Non-streaming
message = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=100,
    messages=[{"role": "user", "content": "What is inference in one sentence?"}]
)
print("Complete response received:")
print(message.content[0].text)
print()

print("=== STREAMING ===")
print("User sees tokens appear one by one:")
print()

# Streaming
with client.messages.stream(
    model="claude-haiku-4-5",
    max_tokens=100,
    messages=[{"role": "user", "content": "What is inference in one sentence?"}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

print()
print()
print("Same response. Streaming feels faster because the user starts reading immediately.")

## My D365 analogy

**Online inference** = posting a single invoice as it arrives.
User submits, system processes, result returned immediately.

**Batch inference** = the period-end batch job.
Everything accumulated over the month processed together overnight.
No user waiting in real time.

**Streaming** = watching a D365 report generate line by line
instead of staring at a loading spinner until the full report appears.
Same data, same processing time — completely different perceived experience.

Both patterns exist in D365.
Both patterns exist in AI systems.
The choice depends on the same question:
does a human need this result immediately, or can it wait?

## LinkedIn post idea

Every D365 consultant knows the difference between a real-time posting
and a batch job run at period end.

Same distinction exists in AI systems.

Online inference: user asks a question, model answers immediately.
Batch inference: 10,000 invoices processed overnight, results ready by morning.

Most people building AI default to real-time for everything.
That is expensive and unnecessary.

A nightly vendor risk score update does not need real-time inference.
A fraud flag on a live transaction does.

Knowing which is which cuts your AI infrastructure costs in half.
The same instinct that makes a good D365 architect
makes a good AI systems designer.